# RBX-AI Studio en Google Colab (gratis, uso personal)

Este notebook monta tu servidor Node.js (RBX-AI Studio v4) en Colab y le da una URL pública con HTTPS usando un túnel Cloudflare gratuito (sin registro).

**Cómo usar:**
1. Sube tu repo a GitHub.
2. En Colab: File → Open notebook → GitHub → pega la URL de tu repo y elige `colab_server.ipynb`.
3. Conecta (botón Connect, arriba a la derecha) y ejecuta la celda de abajo.
4. Cuando aparezca la URL `https://algo.trycloudflare.com`, ábrela en tu navegador y conecta el plugin de Roblox Studio apuntando a esa URL.

**Nota:** la sesión dura hasta ~12 h. Cuando la sesión se cierra, hay que volver a ejecutar la celda (la URL cambia, actualízala en el plugin).

In [ ]:
# Celda 1: preparar el entorno, descargar el proyecto y arrancar el servidor con túnel público
import os, subprocess, threading, time
from google.colab import output

# 1) Clona TU repo de GitHub (cambia la URL por la tuya si prefieres no clonar el repo entero,
    # o usa la URL raw de los 3 archivos: server.js, index.html, package.json)
REPO = "https://github.com/TU_USUARIO/TU_REPO.git"   # <--- PON AQUÍ TU REPO
os.chdir("/content")
!git clone $REPO rbxai 2>/dev/null || mkdir -p rbxai
if not os.path.exists("rbxai/server.js"):
    # Fallback: descargar los 3 archivos sueltos del repo
    BASE = "https://raw.githubusercontent.com/TU_USUARIO/TU_REPO/main/IA%20programacion%20roblox%20studio/"  # <--- TU RUTA
    for f in ["server.js", "index.html", "package.json", "package-lock.json"]:
        !wget -q -O rbxai/$f "$BASE$f"

# 2) API key de OpenRouter (la escribes tú, nunca se guarda en el notebook)
API_KEY = output.eval_js("prompt('Pega tu OPENROUTER_API_KEY de https://openrouter.ai/keys')")
os.environ["OPENROUTER_API_KEY"] = API_KEY
os.environ["PORT"] = "8080"
os.environ["PUBLIC_URL"] = ""   # se rellena con la URL del túnel después

# 3) Instalar dependencias y arrancar el servidor en segundo plano
os.chdir("rbxai")
!npm install --silent

proc = subprocess.Popen(
    ["node", "server.js"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env=os.environ
)

def tail():
    for line in proc.stdout:
        print(line.decode(), end="")

threading.Thread(target=tail, daemon=True).start()
time.sleep(6)
print("✅ Servidor Node.js corriendo en el puerto 8080")

# 4) Túnel Cloudflare gratis (sin registro, sin ngrok)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8080"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

def tail_tunnel():
    for line in tun.stdout:
        print(line.decode(), end="")

threading.Thread(target=tail_tunnel, daemon=True).start()
time.sleep(8)

print()
print("=" * 60)
print("🔗 Abre esta URL en tu navegador (válida mientras la sesión Colab esté activa):")
!curl -s http://localhost:4040/api/tunnels 2>/dev/null || sleep 4
# Cloudflare loguea la URL en su salida; la buscamos:
!curl -s https://localhost:8080 2>/dev/null | head -c 50
